In [1]:
import os,sys,subprocess,glob,importlib,pickle,itertools
from datetime import datetime
import xarray as xr
import numpy as np
import pandas as pd

# Current (ssp370-2025)

In [2]:
obs_files = sorted(glob.glob('/work/bb1152/u290372/REA_output/heat_wEU_JJA/NCAR/CESM2//ssp370-2025-initial/meta/obs/*/*'))

In [5]:
for obs_file in obs_files:
    with xr.open_dataset(obs_file) as nc:  
        nc.sim.values
        member,year = str(nc.sim.values).split('_')

    initial_files = sorted(glob.glob(f'/work/bb1445/u290372/cesm215_archive//BSSP370cmip6_current_2021-2031/*{member}*/atm/hist/*.h1.{year}-*.nc'))

    assert len(initial_files) == 1, "not exactly one file found"

    with xr.open_dataset(initial_files[0]) as nc:
        x = nc['TREFHT']
        x = x.weighted(np.cos(np.radians(x.lat))).mean(('lat','lon')).mean('time')

        ds = xr.Dataset({'GMST':x})
        ds.attrs = nc.attrs
        ds['GMST'].attrs = nc['TREFHT'].attrs
        ds['GMST'].attrs['long_name'] = 'globally averaged surface air temperature (lat weighted)'

        gmst_file = obs_file.replace('obs', 'GMST')
        os.makedirs('/'.join(gmst_file.split('/')[:-1]), exist_ok=True)
        
        ds.to_netcdf(gmst_file)

# piControl